# Etape 2 — Transformation et preparation des donnees



## 1. Imports et chargement des données

On charge le CSV brut et on sépare d'emblée les variables explicatives (`X`) de la cible (`y`).
`Student_ID` est retiré : c'est un identifiant technique sans lien avec le risque, le garder n'apporterait que du bruit (risque de sur-apprentissage sur une colonne unique par ligne).

In [2]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

from preprocessing_utils import IQRCapper

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

ROOT_DIR = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
import sys; sys.path.insert(0, str(ROOT_DIR / "src"))
DATASETPATH = ROOT_DIR / "data" / "raw" / "student_mental_health.csv"
OUTDIR = ROOT_DIR / "data" / "processed"
OUTDIR.mkdir(exist_ok=True)

TARGET = "Mental_Health_Risk"
DROP_COLS = ["Student_ID"]

df = pd.read_csv(DATASETPATH)
df = df.drop(columns=DROP_COLS)

X = df.drop(columns=[TARGET])
y = df[TARGET]

print(f"X : {X.shape} | y : {y.shape}")
X.head()

X : (2000, 15) | y : (2000,)


,Age,Gender,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History
0,22,Male,3,6.6,17.6,2.91,6.9,7.5,Moderate,8.5,4.0,78.0,No,Poor,No
1,21,Male,4,5.6,22.4,2.92,7.6,11.5,Low,4.5,5.3,85.4,Yes,Average,No
2,22,Male,5,7.1,28.1,2.68,6.5,8.9,Low,4.8,6.1,79.5,No,Average,No
3,24,Male,1,4.5,14.8,2.23,5.5,7.6,High,3.7,8.0,73.9,No,Average,Yes
4,20,Female,1,6.7,17.2,2.69,4.3,7.0,Moderate,8.0,7.4,99.8,Yes,NaN,No


## 2. Separation train / test (80 % / 20 %)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Sauvegarde des splits bruts (sans Student_ID) au format CSV dans data/processed/
train_raw_df = pd.concat([X_train, y_train], axis=1)
test_raw_df = pd.concat([X_test, y_test], axis=1)
train_raw_df.to_csv(OUTDIR / "train.csv", index=False)
test_raw_df.to_csv(OUTDIR / "test.csv", index=False)

print(f"Train : {X_train.shape[0]} lignes | Test : {X_test.shape[0]} lignes")
print()
print("Répartition de la cible :")
repartition = pd.DataFrame({
    "train %": (y_train.value_counts(normalize=True) * 100).round(1),
    "test %": (y_test.value_counts(normalize=True) * 100).round(1),
})
repartition


Train : 1600 lignes | Test : 400 lignes

Répartition de la cible :
                    train %  test %
Mental_Health_Risk                 
High                   34.0    34.0
Medium                 33.0    33.0
Low                    33.0    33.0


## 3. Valeurs manquantes — choix de la méthode d'imputation

Rappel de l'EDA : 7 colonnes ont entre ~2 % et ~4,5 % de valeurs manquantes, et uniquement **complètement au hasard** (aucun pattern identifiable). Aucune colonne n'est assez vide pour justifier une suppression (> 40 %).

**Décision :**
- **Variables numériques → imputation par la médiane.** La médiane est robuste aux outliers et aux distributions asymétriques (contrairement à la moyenne) : plusieurs de ces variables ont des outliers IQR détectés à l'EDA, la moyenne serait tirée par ces valeurs extrêmes.
- `Diet_Quality` (seule catégorielle concernée) → **modalité la plus fréquente.** Avec seulement ~3,5 % de manquants, imputer par la modalité dominante (`Average`) déforme très peu la distribution, et cela reste compatible avec son encodage ordinal.

⚠️ Ces imputations sont réalisées **dans le pipeline** (section 8) : les médianes / modes sont appris sur le train seul, jamais sur l'ensemble des données.

Vérification des manquants sur le train :